In [5]:
import pandas as pd
import numpy as np
from google import genai
import os
import json
import time
from dotenv import load_dotenv

load_dotenv("../.env")

GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")
client = genai.Client(api_key=GEMINI_API_KEY)
MODEL = "gemini-2.5-flash"

# Test connection
response = client.models.generate_content(
    model=MODEL,
    contents="Say 'Gemini connected successfully' and nothing else."
)
print(response.text)

Gemini connected successfully


In [6]:
# Load segment summary
rfm = pd.read_csv("../data/processed/rfm_clustered.csv")

segment_summary = rfm.groupby("cluster_label").agg(
    num_customers  = ("CustomerID",  "count"),
    avg_recency    = ("recency",     "mean"),
    avg_frequency  = ("frequency",  "mean"),
    avg_monetary   = ("monetary",   "mean")
).round(2).reset_index()

print(segment_summary)

     cluster_label  num_customers  avg_recency  avg_frequency  avg_monetary
0          At Risk            823        22.94           1.94        475.63
1        Champions            636        11.86          13.60       7192.10
2             Lost           1411       190.66           1.35        342.28
3  Loyal Customers           1050        64.86           4.29       1749.00


In [7]:
# Segment narrative generator
def generate_segment_narrative(segment_row):
    prompt = f"""
You are a senior customer analytics consultant presenting to a C-suite audience.
You have deep expertise in RFM analysis, customer lifetime value, and retention strategy.

You are analyzing a segment from a UK-based e-commerce retailer (gifting/homewares).
The segmentation was built using K-Means clustering on log-transformed, standardized 
RFM features derived from 349,203 transactions across 3,920 customers over 12 months.

SEGMENT DATA:
- Segment Name: {segment_row['cluster_label']}
- Segment Size: {segment_row['num_customers']} customers ({round(segment_row['num_customers']/3920*100, 1)}% of base)
- Avg Recency: {segment_row['avg_recency']} days since last purchase
- Avg Frequency: {segment_row['avg_frequency']} orders per customer
- Avg Monetary Value: £{segment_row['avg_monetary']} lifetime spend

Provide a sharp, data-driven report in exactly 4 sections:

1. SEGMENT PROFILE
   Characterize this segment using the RFM metrics above. Reference the specific 
   numbers. Explain what the combination of recency, frequency, and monetary value 
   tells us about their purchasing behavior and relationship with the brand.

2. REVENUE IMPACT
   Quantify the revenue opportunity or risk. Calculate the estimated total revenue 
   this segment represents (size × monetary). Compare it relative to other segments 
   if relevant. Be precise.

3. CHURN / RETENTION RISK
   Assess the retention risk using recency as the primary signal. Reference 
   industry benchmarks where relevant (e.g. e-commerce average purchase cycle).
   Be direct about whether this segment is deteriorating, stable, or growing.

4. RECOMMENDED ACTIONS
   Provide exactly 3 highly specific, actionable recommendations tailored to this 
   segment's RFM profile. Each action must include:
   - The specific tactic
   - The channel (email, SMS, paid retargeting, etc.)
   - The expected outcome tied to an RFM metric

Tone: executive, data-driven, no filler phrases. Every sentence must add value.
"""
    response = client.models.generate_content(
        model=MODEL,
        contents=prompt
    )
    return response.text


# Test on one segment first
test_row = segment_summary[segment_summary["cluster_label"] == "Champions"].iloc[0]
narrative = generate_segment_narrative(test_row)
print(narrative)

## C-Suite Briefing: Champion Segment Analysis

**Date:** October 26, 2023
**Subject:** Deep Dive into the 'Champions' Customer Segment

This report provides a data-driven analysis of our 'Champions' customer segment, derived from a K-Means clustering of RFM features across 349,203 transactions and 3,920 customers over 12 months.

---

### 1. SEGMENT PROFILE

The 'Champions' segment comprises 636 customers, representing 16.2% of our total customer base. This segment exhibits exceptional purchasing behavior characterized by **high recency, high frequency, and high monetary value**. With an average recency of **11.86 days since last purchase**, these customers have engaged with our brand very recently. Their average frequency of **13.6 orders per customer** over the past 12 months highlights consistent and repeated purchases. Furthermore, an average lifetime monetary value of **£7192.1** demonstrates their significant financial contribution. Collectively, these metrics define the Champio

In [8]:
# Generate narratives for all segments
narratives = {}

for _, row in segment_summary.iterrows():
    print(f"Generating narrative for: {row['cluster_label']}...")
    narratives[row["cluster_label"]] = generate_segment_narrative(row)
    time.sleep(1)  # avoid rate limiting

print("\n--- All narratives generated ---\n")
for segment, narrative in narratives.items():
    print(f"\n{'='*50}")
    print(f"SEGMENT: {segment}")
    print(f"{'='*50}")
    print(narrative)

Generating narrative for: At Risk...
Generating narrative for: Champions...
Generating narrative for: Lost...
Generating narrative for: Loyal Customers...

--- All narratives generated ---


SEGMENT: At Risk
**Customer Segment Analysis: At Risk**

**1. SEGMENT PROFILE**

This segment, named "At Risk," comprises 823 customers, representing 21.0% of our total customer base. Characterized by an average recency of 22.94 days since their last purchase, these customers have engaged with the brand relatively recently. However, their average frequency is low, with only 1.94 orders per customer over the past 12 months. Despite this infrequent purchasing behavior, their average monetary value stands at a significant £475.63 lifetime spend. This combination indicates customers who have historically placed substantial orders but are not frequent purchasers and are now approaching a period of potential inactivity following their last transaction. They are valuable but disengaged between purchases.


In [9]:
llm_code = '''from google import genai
import os
import time
from dotenv import load_dotenv

load_dotenv()

MODEL = "gemini-2.5-flash"
client = genai.Client(api_key=os.getenv("GEMINI_API_KEY"))


def generate_segment_narrative(segment_row, total_customers=3920):
    prompt = f"""
You are a senior customer analytics consultant presenting to a C-suite audience.
You have deep expertise in RFM analysis, customer lifetime value, and retention strategy.

You are analyzing a segment from a UK-based e-commerce retailer (gifting/homewares).
The segmentation was built using K-Means clustering on log-transformed, standardized
RFM features derived from 349,203 transactions across 3,920 customers over 12 months.

SEGMENT DATA:
- Segment Name: {segment_row["cluster_label"]}
- Segment Size: {segment_row["num_customers"]} customers ({round(segment_row["num_customers"]/total_customers*100, 1)}% of base)
- Avg Recency: {segment_row["avg_recency"]} days since last purchase
- Avg Frequency: {segment_row["avg_frequency"]} orders per customer
- Avg Monetary Value: £{segment_row["avg_monetary"]} lifetime spend

Provide a sharp, data-driven report in exactly 4 sections:

1. SEGMENT PROFILE
   Characterize this segment using the RFM metrics above. Reference the specific
   numbers. Explain what the combination of recency, frequency, and monetary value
   tells us about their purchasing behavior and relationship with the brand.

2. REVENUE IMPACT
   Quantify the revenue opportunity or risk. Calculate the estimated total revenue
   this segment represents (size x monetary). Compare it relative to other segments
   if relevant. Be precise.

3. CHURN / RETENTION RISK
   Assess the retention risk using recency as the primary signal. Reference
   industry benchmarks where relevant (e.g. e-commerce average purchase cycle).
   Be direct about whether this segment is deteriorating, stable, or growing.

4. RECOMMENDED ACTIONS
   Provide exactly 3 highly specific, actionable recommendations tailored to this
   segment\'s RFM profile. Each action must include:
   - The specific tactic
   - The channel (email, SMS, paid retargeting, etc.)
   - The expected outcome tied to an RFM metric

Tone: executive, data-driven, no filler phrases. Every sentence must add value.
"""
    response = client.models.generate_content(
        model=MODEL,
        contents=prompt
    )
    return response.text


def generate_all_narratives(segment_summary_df, total_customers=3920, sleep=1):
    narratives = {}
    for _, row in segment_summary_df.iterrows():
        narratives[row["cluster_label"]] = generate_segment_narrative(
            row, total_customers
        )
        time.sleep(sleep)
    return narratives
'''

with open("../src/llm.py", "w") as f:
    f.write(llm_code)

print("src/llm.py written successfully")

src/llm.py written successfully


In [10]:
import sys
sys.path.append("../src")

from llm import generate_all_narratives

narratives = generate_all_narratives(segment_summary)
print("Module works correctly")
for seg in narratives:
    print(f"  {seg}: {len(narratives[seg])} chars")

Module works correctly
  At Risk: 4091 chars
  Champions: 3915 chars
  Lost: 4294 chars
  Loyal Customers: 3296 chars


In [12]:
import json
with open("../data/processed/segment_narratives.json", "w") as f:
    json.dump(narratives, f, indent=2)

print("Saved: data/processed/segment_narratives.json")

Saved: data/processed/segment_narratives.json
